# AutoGen 基礎對話教程

## 📚 簡介

AutoGen 是微軟推出的多 Agent 對話框架，能夠實現：
- 多個 AI Agent 之間的自動對話
- 人機協作
- 自動化程式碼執行
- 複雜任務分解和協作

本教程將帶你創建第一個 AutoGen 對話系統。

## 1. 安裝 AutoGen

In [ ]:
# 安裝 AutoGen
!pip install pyautogen -q

## 2. 配置 API

In [ ]:
import os
from dotenv import load_dotenv

# 加載環境變數
load_dotenv()

# 配置 LLM
config_list = [
    {
        "model": "gpt-4",
        "api_key": os.getenv("OPENAI_API_KEY")
    }
]

llm_config = {
    "config_list": config_list,
    "temperature": 0.7
}

print("配置完成！")

## 3. 創建第一個 Assistant Agent

Assistant Agent 是一個能夠理解和回答問題的 AI 助手。

In [ ]:
from autogen import AssistantAgent

# 創建 Assistant Agent
assistant = AssistantAgent(
    name="助理",
    system_message="你是一個樂於助人的 AI 助理，擅長回答問題和提供建議。",
    llm_config=llm_config
)

print("Assistant Agent 已創建！")

## 4. 創建 User Proxy Agent

User Proxy Agent 代表用戶，可以執行程式碼和與 Assistant 對話。

In [ ]:
from autogen import UserProxyAgent

# 創建 User Proxy Agent
user_proxy = UserProxyAgent(
    name="用戶代理",
    human_input_mode="NEVER",  # 不需要人工輸入
    max_consecutive_auto_reply=5,
    code_execution_config={
        "work_dir": "coding",
        "use_docker": False  # 不使用 Docker
    }
)

print("User Proxy Agent 已創建！")

## 5. 簡單對話範例

讓兩個 Agent 進行簡單對話：

In [ ]:
# 開始對話
user_proxy.initiate_chat(
    assistant,
    message="請向我介紹什麼是人工智慧？請用三個要點說明。"
)

## 6. 程式碼生成範例

讓 Agent 幫我們寫程式碼：

In [ ]:
# 創建能執行程式碼的 User Proxy
code_executor = UserProxyAgent(
    name="程式碼執行器",
    human_input_mode="NEVER",
    max_consecutive_auto_reply=10,
    is_termination_msg=lambda x: x.get("content", "").rstrip().endswith("TERMINATE"),
    code_execution_config={
        "work_dir": "coding",
        "use_docker": False
    }
)

# 請求生成和執行程式碼
code_executor.initiate_chat(
    assistant,
    message="""請寫一個 Python 函數來計算斐波那契數列的第 n 項，
    然後計算第 10 項的值。請執行程式碼並顯示結果。"""
)

## 7. 數學問題解決

使用 AutoGen 解決數學問題：

In [ ]:
# 數學問題
user_proxy.initiate_chat(
    assistant,
    message="""請幫我解決這個問題：
    一個長方形的長是寬的 2 倍，如果周長是 60 公分，
    請計算長方形的面積是多少平方公分？
    請寫 Python 代碼來計算。"""
)

## 8. 自定義系統訊息

為 Agent 設定不同的角色：

In [ ]:
# 創建一個數學老師 Agent
math_teacher = AssistantAgent(
    name="數學老師",
    system_message="""你是一位經驗豐富的數學老師。
    你會用簡單易懂的方式解釋數學概念，
    並提供步驟清晰的解題方法。""",
    llm_config=llm_config
)

# 創建學生 Agent
student = UserProxyAgent(
    name="學生",
    human_input_mode="NEVER",
    max_consecutive_auto_reply=3,
    code_execution_config=False  # 學生不執行程式碼
)

# 學生向老師提問
student.initiate_chat(
    math_teacher,
    message="老師，請問什麼是質數？能舉幾個例子嗎？"
)

## 9. 對話終止條件

設定對話何時結束：

In [ ]:
# 自定義終止條件
def is_termination_msg(msg):
    """檢查是否應該終止對話"""
    content = msg.get("content", "")
    return "完成" in content or "TERMINATE" in content

# 創建帶終止條件的 Agent
user_with_termination = UserProxyAgent(
    name="用戶",
    human_input_mode="NEVER",
    max_consecutive_auto_reply=10,
    is_termination_msg=is_termination_msg,
    code_execution_config=False
)

print("帶終止條件的 Agent 已創建")

## 10. 對話歷史查看

In [ ]:
# 查看對話歷史
print("對話歷史：")
for i, msg in enumerate(user_proxy.chat_messages[assistant], 1):
    print(f"\n消息 {i}:")
    print(f"角色: {msg.get('role', 'unknown')}")
    print(f"內容: {msg.get('content', '')[:100]}...")

## 📝 總結

本教程中，我們學習了：

1. ✅ AutoGen 的安裝和配置
2. ✅ 創建 AssistantAgent 和 UserProxyAgent
3. ✅ 基本對話流程
4. ✅ 程式碼生成和執行
5. ✅ 自定義 Agent 角色
6. ✅ 設定終止條件
7. ✅ 查看對話歷史

## 🎯 下一步

- 學習多 Agent 協作：`1.多Agent協作.ipynb`
- 探索程式碼執行功能：`2.程式碼執行Agent.ipynb`

## 🔗 資源

- [AutoGen 官方文檔](https://microsoft.github.io/autogen/)
- [GitHub](https://github.com/microsoft/autogen)